# Lab 09: Length and Distance

## Measuring size, difference, error, and similarity

This lab supports Chapter 9 of *Linear Algebra: A Story of Data, Geometry, and Intelligence*.

The goal is not only to compute norms. The goal is to understand how a choice of distance changes how we compare data.

You will explore:

- Euclidean norm and distance
- distance as prediction error
- feature scaling and standardization
- nearest neighbors
- image distance
- pairwise distance matrices
- high-dimensional distance behavior
- how matrices change distances


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

def norm2(v):
    return np.sqrt(np.sum(v**2))

def euclidean_distance(x, y):
    return norm2(x - y)

print("Ready.")


## 1. The length of a vector

The Euclidean norm is

$$
\|v\|_2=\sqrt{v_1^2+v_2^2+\cdots+v_n^2}.
$$

In two dimensions, this is the Pythagorean theorem. In higher dimensions, it is the same idea extended coordinate by coordinate.


In [ ]:
v = np.array([3, 4])
print("v =", v)
print("manual norm =", norm2(v))
print("NumPy norm =", np.linalg.norm(v))


In [ ]:
plt.figure(figsize=(6,6))
plt.quiver(0, 0, v[0], v[1], angles="xy", scale_units="xy", scale=1)
plt.plot([0, v[0]], [0, 0], linestyle="--")
plt.plot([v[0], v[0]], [0, v[1]], linestyle="--")
plt.text(1.5, -0.35, "3")
plt.text(3.15, 2, "4")
plt.text(1.3, 2.35, "length = 5")
plt.axhline(0)
plt.axvline(0)
plt.xlim(-1,5)
plt.ylim(-1,5)
plt.grid(True)
plt.gca().set_aspect("equal", adjustable="box")
plt.title("The norm of (3,4)")
plt.show()


### Student task

Change `v` to several different vectors. Try vectors with negative entries. Does the norm ever become negative? Why not?


In [ ]:
# TODO: Change this vector and recompute its norm.
v = np.array([-5, 12])
print("v =", v)
print("norm =", np.linalg.norm(v))


## 2. Distance is length of difference

For two vectors $x$ and $y$,

$$
d(x,y)=\|x-y\|_2.
$$

The difference vector tells us how to move from one point to the other.


In [ ]:
x = np.array([1, 2])
y = np.array([5, 5])

diff = y - x
print("x =", x)
print("y =", y)
print("y - x =", diff)
print("distance =", np.linalg.norm(diff))


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter([x[0], y[0]], [x[1], y[1]], s=100)
plt.quiver(x[0], x[1], diff[0], diff[1], angles="xy", scale_units="xy", scale=1)
plt.text(x[0]-0.25, x[1]-0.3, "x")
plt.text(y[0]+0.1, y[1], "y")
plt.axhline(0)
plt.axvline(0)
plt.xlim(0,6)
plt.ylim(0,6)
plt.grid(True)
plt.gca().set_aspect("equal", adjustable="box")
plt.title("Distance between two points")
plt.show()


## 3. Distance as prediction error

If $y$ is the true vector and $\hat y$ is a prediction, then

$$
e=y-\hat y
$$

is the error vector. Its norm is a total error size.


In [ ]:
y_true = np.array([10, 12, 9, 15, 11, 13])
y_pred = np.array([11, 10, 10, 14, 12, 13.5])

error = y_true - y_pred
sse = np.sum(error**2)
rmse = np.sqrt(np.mean(error**2))

print("error vector =", error)
print("Euclidean error =", np.linalg.norm(error))
print("sum of squared errors =", sse)
print("RMSE =", rmse)


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(y_true, marker="o", label="true")
plt.plot(y_pred, marker="o", label="predicted")
for i in range(len(y_true)):
    plt.plot([i, i], [y_true[i], y_pred[i]], linestyle="--")
plt.title("Prediction errors as vertical gaps")
plt.xlabel("index")
plt.ylabel("value")
plt.grid(True)
plt.legend()
plt.show()


### Student task

Create a second prediction vector. Which prediction has smaller RMSE? Which one has smaller maximum error?


In [ ]:
# TODO: Modify this prediction vector.
y_pred_2 = np.array([10.5, 11.5, 9.5, 15.5, 10.5, 12.5])
error_2 = y_true - y_pred_2

print("RMSE model 1 =", np.sqrt(np.mean((y_true - y_pred)**2)))
print("RMSE model 2 =", np.sqrt(np.mean(error_2**2)))
print("max error model 1 =", np.max(np.abs(y_true - y_pred)))
print("max error model 2 =", np.max(np.abs(error_2)))


## 4. Feature scaling changes distance

Distances depend on units. A feature measured in dollars may dominate a feature measured in miles.

We will use a small apartment dataset:

- rent in dollars
- size in square feet
- distance to campus in miles
- bedrooms


In [ ]:
apartments = np.array([
    [2400, 800, 1.2, 2],
    [2600, 760, 0.8, 2],
    [1800, 600, 3.5, 1],
    [3200, 1100, 0.5, 3],
    [2100, 700, 2.2, 2],
    [2300, 850, 1.8, 2]
], dtype=float)

names = np.array(["A", "B", "C", "D", "E", "F"])

query = np.array([2500, 780, 1.0, 2], dtype=float)
raw_distances = np.linalg.norm(apartments - query, axis=1)

for name, d in zip(names, raw_distances):
    print(name, d)
print("nearest by raw distance:", names[np.argmin(raw_distances)])


In [ ]:
mu = apartments.mean(axis=0)
sigma = apartments.std(axis=0, ddof=0)

apartments_z = (apartments - mu) / sigma
query_z = (query - mu) / sigma

z_distances = np.linalg.norm(apartments_z - query_z, axis=1)

for name, d in zip(names, z_distances):
    print(name, d)
print("nearest by standardized distance:", names[np.argmin(z_distances)])


In [ ]:
plt.figure(figsize=(8,4))
indices = np.arange(len(names))
width = 0.35
plt.bar(indices - width/2, raw_distances / raw_distances.max(), width, label="raw distance, rescaled")
plt.bar(indices + width/2, z_distances / z_distances.max(), width, label="standardized distance, rescaled")
plt.xticks(indices, names)
plt.ylabel("distance divided by maximum")
plt.title("Raw and standardized distances can rank neighbors differently")
plt.legend()
plt.grid(axis="y")
plt.show()


### Reflection

Which feature dominates the raw distance? How can you tell?


## 5. Comparing distance rules

Let $r=x-y$ be a difference vector.

Common norms include:

$$
\|r\|_1=\sum_i |r_i|,
\qquad
\|r\|_2=\sqrt{\sum_i r_i^2},
\qquad
\|r\|_\infty=\max_i |r_i|.
$$


In [ ]:
x = np.array([1, 2, 8, 4])
y = np.array([4, 4, 2, 5])
r = x - y

l1 = np.sum(np.abs(r))
l2 = np.linalg.norm(r)
linf = np.max(np.abs(r))

print("difference =", r)
print("L1 distance =", l1)
print("L2 distance =", l2)
print("L-infinity distance =", linf)


## 6. Pairwise distance matrices

A pairwise distance matrix stores every distance between every pair of points.


In [ ]:
def distance_matrix(X):
    n = X.shape[0]
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D[i, j] = np.linalg.norm(X[i] - X[j])
    return D

np.random.seed(4)
X = np.random.normal(size=(10, 2))
D = distance_matrix(X)
print(np.round(D, 2))


In [ ]:
plt.figure(figsize=(6,5))
plt.imshow(D)
plt.colorbar(label="distance")
plt.title("Pairwise distance matrix")
plt.xlabel("point index")
plt.ylabel("point index")
plt.show()


### Student task

Explain why the diagonal is zero and why the matrix is symmetric.


## 7. Nearest-neighbor classification

Nearest-neighbor classification uses distance to classify a new point.


In [ ]:
np.random.seed(12)
class0 = np.random.normal(loc=[0, 0], scale=0.7, size=(40, 2))
class1 = np.random.normal(loc=[2.5, 2], scale=0.7, size=(40, 2))
X_train = np.vstack([class0, class1])
y_train = np.array([0]*len(class0) + [1]*len(class1))

new_point = np.array([1.7, 1.1])
d = np.linalg.norm(X_train - new_point, axis=1)
nearest = np.argmin(d)

plt.figure(figsize=(7,6))
plt.scatter(class0[:,0], class0[:,1], label="class 0")
plt.scatter(class1[:,0], class1[:,1], label="class 1")
plt.scatter(new_point[0], new_point[1], marker="*", s=220, label="new point")
plt.plot([new_point[0], X_train[nearest,0]], [new_point[1], X_train[nearest,1]], linestyle="--")
plt.legend()
plt.grid(True)
plt.gca().set_aspect("equal", adjustable="box")
plt.title("One-nearest-neighbor classification")
plt.show()

print("nearest class =", y_train[nearest])
print("nearest distance =", d[nearest])


## 8. Image distance

We can flatten a small image into a vector. Then image distance becomes vector distance.


In [ ]:
def make_square(shift_x=0, shift_y=0):
    img = np.zeros((12,12))
    img[3+shift_y:8+shift_y, 3+shift_x:8+shift_x] = 1
    return img

img_a = make_square(0, 0)
img_b = make_square(2, 1)
img_c = make_square(0, 0)
img_c[3:8, 3:8] = 0.6

images = [img_a, img_b, img_c]
titles = ["A", "B: shifted", "C: dimmer"]

fig, axes = plt.subplots(1, 3, figsize=(9,3))
for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
plt.show()

vectors = [img.reshape(-1) for img in images]
for i in range(3):
    for j in range(i+1, 3):
        print(f"distance {titles[i]} to {titles[j]} =", np.linalg.norm(vectors[i] - vectors[j]))


### Reflection

Which image is closer to A by pixel distance? Does that match your visual intuition?


## 9. High-dimensional distance

Random points in high dimensions tend to become far apart, and distances often concentrate.


In [ ]:
np.random.seed(1)
dimensions = [2, 5, 10, 20, 50, 100, 200, 500]
means = []
stds = []
rel = []

for d in dimensions:
    X = np.random.normal(size=(1000, d))
    Y = np.random.normal(size=(1000, d))
    distances = np.linalg.norm(X - Y, axis=1)
    means.append(distances.mean())
    stds.append(distances.std())
    rel.append(distances.std() / distances.mean())

plt.figure(figsize=(7,4))
plt.plot(dimensions, means, marker="o", label="mean")
plt.plot(dimensions, stds, marker="o", label="standard deviation")
plt.xlabel("dimension")
plt.ylabel("distance")
plt.title("Distance growth in high dimensions")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(7,4))
plt.plot(dimensions, rel, marker="o")
plt.xlabel("dimension")
plt.ylabel("std / mean")
plt.title("Relative spread of distances")
plt.grid(True)
plt.show()


### Student task

Repeat the experiment with random points drawn uniformly from $[0,1]^d$. What changes? What stays similar?


In [ ]:
# TODO: Repeat the high-dimensional experiment using np.random.rand.
# Store and plot the mean distance and relative spread.


## 10. Matrices change distances

A matrix transformation sends $x$ to $Ax$. The distance between two transformed points is

$$
\|Ax-Ay\|=\|A(x-y)\|.
$$

Some matrices preserve distance. Some stretch it. Some collapse it.


In [ ]:
theta = np.pi / 6
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
S = np.array([[2.0, 0],
              [0, 0.5]])
C = np.array([[1.0, 0],
              [0, 0]])

x = np.array([1, 1])
y = np.array([3, 2])

for name, A in [("rotation", R), ("stretch/squeeze", S), ("collapse", C)]:
    before = np.linalg.norm(x - y)
    after = np.linalg.norm(A @ x - A @ y)
    print(name)
    print("before =", before)
    print("after  =", after)
    print("ratio  =", after / before if before != 0 else np.nan)
    print()


In [ ]:
def plot_transform(A, title):
    pts = np.array([[0,0], [1,0], [1,1], [0,1], [0,0]], dtype=float)
    tpts = pts @ A.T
    plt.figure(figsize=(6,6))
    plt.plot(pts[:,0], pts[:,1], marker="o", label="original square")
    plt.plot(tpts[:,0], tpts[:,1], marker="o", label="transformed")
    plt.axhline(0)
    plt.axvline(0)
    plt.grid(True)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.title(title)
    plt.legend()
    plt.show()

plot_transform(R, "Rotation preserves distances")
plot_transform(S, "Stretch/squeeze changes distances")
plot_transform(C, "Collapse destroys some distances")


## Final reflection

Write a short paragraph answering these questions:

1. What does distance mean in a vector space?
2. Why can distance be misleading in real datasets?
3. What is one practical use of distance in machine learning?
4. What is one warning about high-dimensional distance?
